<a href="https://colab.research.google.com/github/uniesecruz/ESALQ_money_laundering/blob/HI-medium/TCC_ESALQ_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Instalação das bibliotecas

In [2]:
# Instalar pyspark
!pip install pyspark

# Leitura do arquivos CSV com spark

In [3]:
from pyspark.sql import SparkSession

# Ajustado para os 167GB reais da sua instância
memory_limit = "140g"

spark = SparkSession.builder \
    .appName("TCC_AML_HighPerformance") \
    .config("spark.driver.memory", memory_limit) \
    .config("spark.executor.memory", memory_limit) \
    .config("spark.driver.maxResultSize", "30g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

print(f"Spark inicializado com {memory_limit} de RAM disponível.")

Spark inicializado com 140g de RAM disponível.


In [ ]:
# # Carregar o arquivo CSV usando Spark
# spark_trans_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Medium_Trans.csv', header=True, inferSchema=True)

# print("Schema do Spark DataFrame de Transações:")
# spark_trans_df.printSchema()

# print("Primeiras 5 linhas do Spark DataFrame de Transações:")
# spark_trans_df.show(5)

Schema do Spark DataFrame de Transações:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Transações:
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+--------------

In [ ]:
# # Carregar o arquivo CSV de contas usando Spark
# spark_accounts_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Medium_accounts.csv', header=True, inferSchema=True)

# print("Schema do Spark DataFrame de Contas:")
# spark_accounts_df.printSchema()

# print("Primeiras 5 linhas do Spark DataFrame de Contas:")
# spark_accounts_df.show(5)

Schema do Spark DataFrame de Contas:
root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Contas:
+--------------------+-------+--------------+-----------+--------------------+
|           Bank Name|Bank ID|Account Number|  Entity ID|         Entity Name|
+--------------------+-------+--------------+-----------+--------------------+
|     China Bank #561|  53267|     817D00980|2AA1F24F180| Corporation #183669|
|   Spain Bank #18657| 316997|     808BB2280|2AA1EEB8540| Partnership #193780|
|First Bank of Helena| 339367|     8505ED380|2AA206D7790|Sole Proprietorsh...|
|   Mexico Bank #3367|3148419|     8363D4180|2AA2001B1A0| Partnership #133577|
|Switzerland Bank ...|3174937|     842090C80|2AA20224CB0|Sole Proprietorsh...|
+--------------------+-------+--------------+-----------+--------

Salvando os arquivos em parquet

In [ ]:
# spark_trans_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_Trans', mode='overwrite')
# print("spark_trans_df salvo em parquet.")

# spark_accounts_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_accounts', mode='overwrite')
# print("spark_accounts_df salvo em parquet.")

spark_trans_df salvo em parquet.
spark_accounts_df salvo em parquet.


# Lendo o arquivo parquet (Começar aqui)

In [ ]:
# from pyspark.sql import SparkSession
# import os

# # 1. Definir a memória máxima para o driver (essencial no Colab)
# # Deixamos uma margem de segurança para o Sistema Operacional (~4-5GB)
# memory_limit = "46g"

# spark = SparkSession.builder \
#     .appName("Max_Performance_Spark") \
#     .config("spark.driver.memory", memory_limit) \
#     .config("spark.executor.memory", memory_limit) \
#     .config("spark.driver.maxResultSize", "10g") \
#     .config("spark.sql.shuffle.partitions", "200") \
#     .config("spark.memory.fraction", "0.8") \
#     .config("spark.memory.storageFraction", "0.3") \
#     .config("spark.ui.port", "4050") \
#     .getOrCreate()

# print(f"SparkSession inicializada com foco em High-RAM ({memory_limit}).")

SparkSession inicializada com foco em High-RAM (46g).


In [ ]:
spark_trans_df= spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_Trans')
spark_accounts_df = spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Medium_accounts')

# Profiling Trans

In [ ]:
print("### Profiling do spark_trans_df ###")

print("\nSchema do DataFrame:")
spark_trans_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_trans_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_trans_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_trans_df.summary().show()

print(f"\nNúmero total de linhas: {spark_trans_df.count()}")

print("\nNomes das Colunas:")
print(spark_trans_df.columns)

### Profiling do spark_trans_df ###

Schema do DataFrame:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)


Tipos de Dados das Colunas:
Timestamp: string
From Bank: int
Account2: string
To Bank: int
Account4: string
Amount Received: double
Receiving Currency: string
Amount Paid: double
Payment Currency: string
Payment Format: string
Is Laundering: int

Estatísticas Descritivas (describe()):
+-------+----------------+-----------------+---------+------------------+---------+--------------------+------------------+--------------------+-----------------+----

# Profiling Accounts

In [ ]:
print("### Profiling do spark_accounts_df ###")

print("\nSchema do DataFrame:")
spark_accounts_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_accounts_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_accounts_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_accounts_df.summary().show()

print(f"\nNúmero total de linhas: {spark_accounts_df.count()}")

print("\nNomes das Colunas:")
print(spark_accounts_df.columns)

### Profiling do spark_accounts_df ###

Schema do DataFrame:
root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)


Tipos de Dados das Colunas:
Bank Name: string
Bank ID: int
Account Number: string
Entity ID: string
Entity Name: string

Estatísticas Descritivas (describe()):
+-------+------------------+-----------------+--------------+-----------+--------------------+
|summary|         Bank Name|          Bank ID|Account Number|  Entity ID|         Entity Name|
+-------+------------------+-----------------+--------------+-----------+--------------------+
|  count|           2087786|          2087786|       2087786|    2087786|             2087786|
|   mean|              NULL|632102.3694339362|      Infinity|       NULL|                NULL|
| stddev|              NULL|970374.5529627781|           NaN|       NULL|             

In [ ]:
spark_trans_df =spark_trans_df.withColumnRenamed("Account2", "From Account") \
                                     .withColumnRenamed("Account4", "To Account")

# Join dos datasets

In [ ]:

from pyspark.sql import functions as F



# 2. Renomear colunas de spark_trans_df (Equivalente ao spark_trans_df.columns no Pandas)
# Nota: Como o Spark não aceita atribuição direta de lista em .columns, usamos select e alias
new_columns = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account',
               'Amount Received', 'Receiving Currency', 'Amount Paid',
               'Payment Currency', 'Payment Format', 'Is Laundering']

spark_trans_df = spark_trans_df.toDF(*new_columns)

# 3. Preparar spark_accounts_df para os joins (selecionando apenas colunas necessárias)
# Isso evita colisões de nomes e facilita o mapeamento posterior
acc_info = spark_accounts_df.select(
    F.col("Bank ID"),
    F.col("Account Number"),
    F.col("Bank Name"),
    F.col("Entity ID"),
    F.col("Entity Name")
)

# --- JOIN 1: Informações da conta de ORIGEM (Sender) ---
trans_enriched_df = spark_trans_df.join(
    F.broadcast(acc_info),
    (spark_trans_df["From Bank"] == acc_info["Bank ID"]) &
    (spark_trans_df["From Account"] == acc_info["Account Number"]),
    how='left'
).select(
    spark_trans_df["*"], # Mantém todas as colunas originais da transação
    F.col("Bank Name").alias("From Bank Name"),
    F.col("Entity ID").alias("From Entity ID"),
    F.col("Entity Name").alias("From Entity Name")
)

# --- JOIN 2: Informações da conta de DESTINO (Receiver) ---
trans_enriched_df = trans_enriched_df.join(
    F.broadcast(acc_info),
    (trans_enriched_df["To Bank"] == acc_info["Bank ID"]) &
    (trans_enriched_df["To Account"] == acc_info["Account Number"]),
    how='left'
).select(
    trans_enriched_df["*"], # Mantém as colunas do primeiro join
    F.col("Bank Name").alias("To Bank Name"),
    F.col("Entity ID").alias("To Entity ID"),
    F.col("Entity Name").alias("To Entity Name")
)

# Exibir o resultado
print("Tabela de Transações Enriquecida:")
trans_enriched_df.show(5)

# Se precisar contar o total (equivalente ao print final)
# print(f"Total de registros: {trans_enriched_df.count()}")

Tabela de Transações Enriquecida:
+----------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+--------------------+--------------+--------------------+--------------------+------------+------------------+
|       Timestamp|From Bank|From Account|To Bank|To Account|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|      From Bank Name|From Entity ID|    From Entity Name|        To Bank Name|To Entity ID|    To Entity Name|
+----------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+--------------------+--------------+--------------------+--------------------+------------+------------------+
|2022/09/01 11:02|   207628|   84CD42740| 207628| 84CD42740|           3.46|         US Dollar|       3.46|       US Dollar|  Reinvestment|            0|Willows Savings Bank|

# Salvando base enriquecida em parquet

In [ ]:
# trans_enriched_df.write.parquet('/content/drive/MyDrive/TCC/data/processed/parquet/HI-Medium_enriched', mode='overwrite')
# print("trans_enriched_df salvo em parquet.")

trans_enriched_df salvo em parquet.


Realizando Leitura da base enriquecida em parquet

In [ ]:
trans_enriched_df = spark.read.parquet('/content/drive/MyDrive/TCC/data/processed/parquet/HI-Medium_enriched')

In [ ]:
from dataclasses import dataclass, field
from typing import Dict
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql import functions as F

SECONDS_BY_WINDOW = {
    "1h": 3600,
    "24h": 86400,
    "7d": 604800,
    "30d": 2592000,
}

@dataclass
class FeatureEngineeringPipelineSpark:
    """Generate AML features using deterministic Spark window operations."""

    timestamp_col: str = "Timestamp"
    account_col: str = "From Account"
    amount_col: str = "Amount Received"
    bank_col: str = "Receiving Currency"
    country_col: str = "From Bank"
    velocity_windows: Dict[str, str] = field(
        default_factory=lambda: {"1h": "1h", "24h": "24h", "7d": "7d"}
    )
    ratio_window: str = "30d"
    smurf_threshold: float = 10000.0

    def fit(self, df: DataFrame) -> "FeatureEngineeringPipelineSpark":
        """No learned state; kept for API compatibility."""
        _ = df
        return self

    def transform(self, df: DataFrame) -> DataFrame:
        """Apply full feature engineering pipeline in Spark."""
        base = (
            df.withColumn(self.timestamp_col, F.to_timestamp(F.col(self.timestamp_col), 'yyyy/MM/dd HH:mm'))
            .filter(F.col(self.timestamp_col).isNotNull())
            .withColumn("_ts_long", F.col(self.timestamp_col).cast("long"))
        )

        # Deterministic ordering for lag-based features.
        order_cols = [
            F.col(self.timestamp_col).asc(),
            F.col(self.amount_col).asc_nulls_last(),
            F.col("To Account").asc_nulls_last(),
        ]
        row_win = Window.partitionBy(self.account_col).orderBy(*order_cols)

        enriched = self._add_velocity_features(base)
        enriched = self._add_ratio_features(enriched)
        enriched = self._add_behavioral_features(enriched, row_win, order_cols)
        enriched = self._add_smurfing_features(enriched)

        enriched = enriched.drop("_ts_long")
        return enriched

    def fit_transform(self, df: DataFrame) -> DataFrame:
        return self.fit(df).transform(df)

    def _add_velocity_features(self, df: DataFrame) -> DataFrame:
        out = df
        for window_name in self.velocity_windows:
            seconds = SECONDS_BY_WINDOW[window_name]
            win = (
                Window.partitionBy(self.account_col)
                .orderBy(F.col("_ts_long"))
                .rangeBetween(-seconds, -1)
            )
            out = out.withColumn(
                f"txn_count_{window_name}_velocity",
                F.coalesce(F.count(F.lit(1)).over(win), F.lit(0)).cast("double"),
            )
            out = out.withColumn(
                f"amount_sum_{window_name}_velocity",
                F.coalesce(F.sum(F.col(self.amount_col)).over(win), F.lit(0.0)),
            )
            out = out.withColumn(
                f"amount_mean_{window_name}_velocity",
                F.coalesce(F.avg(F.col(self.amount_col)).over(win), F.lit(0.0)),
            )
            out = out.withColumn(
                f"amount_max_{window_name}_velocity",
                F.coalesce(F.max(F.col(self.amount_col)).over(win), F.lit(0.0)),
            )
            if window_name == "7d":
                out = out.withColumn(
                    f"amount_std_{window_name}_velocity",
                    F.coalesce(F.stddev(F.col(self.amount_col)).over(win), F.lit(0.0)),
                )

        return out

    def _add_ratio_features(self, df: DataFrame) -> DataFrame:
        seconds = SECONDS_BY_WINDOW[self.ratio_window]
        win = (
            Window.partitionBy(self.account_col)
            .orderBy(F.col("_ts_long"))
            .rangeBetween(-seconds, -1)
        )

        hist_mean = F.avg(F.col(self.amount_col)).over(win)
        hist_max = F.max(F.col(self.amount_col)).over(win)
        hist_std = F.stddev(F.col(self.amount_col)).over(win)

        out = df.withColumn("_hist_mean", hist_mean)
        out = out.withColumn("_hist_max", hist_max)
        out = out.withColumn("_hist_std", hist_std)

        out = out.withColumn(
            "amount_to_historical_mean_ratio",
            F.when(F.col("_hist_mean") > 0, F.col(self.amount_col) / F.col("_hist_mean")).otherwise(0.0),
        )
        out = out.withColumn(
            "amount_to_historical_max_ratio",
            F.when(F.col("_hist_max") > 0, F.col(self.amount_col) / F.col("_hist_max")).otherwise(0.0),
        )
        out = out.withColumn(
            "amount_zscore_historical",
            F.when(F.col("_hist_std") > 0, (F.col(self.amount_col) - F.col("_hist_mean")) / F.col("_hist_std")).otherwise(0.0),
        )

        return out.drop("_hist_mean", "_hist_max", "_hist_std")

    def _add_behavioral_features(
        self,
        df: DataFrame,
        row_win: Window,
        order_cols: list,
    ) -> DataFrame:
        out = df

        lag_ts = F.lag(F.col("_ts_long")).over(row_win)
        out = out.withColumn(
            "time_since_last_txn_seconds",
            F.coalesce((F.col("_ts_long") - lag_ts).cast("double"), F.lit(0.0)),
        )

        lag_bank = F.lag(F.col(self.bank_col)).over(row_win)
        out = out.withColumn(
            "bank_change_flag",
            F.when(lag_bank.isNull(), F.lit(0)).when(F.col(self.bank_col) != lag_bank, F.lit(1)).otherwise(F.lit(0)),
        )

        # 1 if this is the first time account->country pair appears.
        country_win = Window.partitionBy(self.account_col, self.country_col).orderBy(*order_cols)
        out = out.withColumn(
            "is_new_country",
            F.when(F.row_number().over(country_win) == 1, F.lit(1)).otherwise(F.lit(0)),
        )

        out = out.withColumn("hour_of_day", F.hour(F.col(self.timestamp_col)))
        out = out.withColumn(
            "is_unusual_hour",
            F.when((F.col("hour_of_day") < 6) | (F.col("hour_of_day") > 22), F.lit(1)).otherwise(F.lit(0)),
        )

        return out

    def _add_smurfing_features(self, df: DataFrame) -> DataFrame:
        win24 = (
            Window.partitionBy(self.account_col)
            .orderBy(F.col("_ts_long"))
            .rangeBetween(-SECONDS_BY_WINDOW["24h"], -1)
        )

        is_smurf = (F.col(self.amount_col) >= self.smurf_threshold * 0.8) & (
            F.col(self.amount_col) < self.smurf_threshold
        )
        smurf_amount = F.when(is_smurf, F.col(self.amount_col)).otherwise(F.lit(0.0))
        proximity = F.when(
            F.col(self.amount_col) < self.smurf_threshold,
            (F.lit(self.smurf_threshold) - F.col(self.amount_col)) / F.lit(self.smurf_threshold),
        ).otherwise(F.lit(0.0))

        out = df.withColumn(
            "smurf_txn_count_24h_behavioral",
            F.coalesce(F.sum(F.when(is_smurf, 1).otherwise(0)).over(win24).cast("double"), F.lit(0.0)),
        )
        out = out.withColumn(
            "smurf_amount_sum_24h_behavioral",
            F.coalesce(F.sum(smurf_amount).over(win24), F.lit(0.0)),
        )
        out = out.withColumn(
            "smurf_proximity_score_behavioral",
            F.coalesce(F.avg(proximity).over(win24), F.lit(0.0)),
        )

        return out

# Instantiate the feature engineering pipeline
feature_pipeline = FeatureEngineeringPipelineSpark(
    timestamp_col="Timestamp",
    account_col="From Account",
    amount_col="Amount Received",
    bank_col="Receiving Currency",
    country_col="From Bank",
    velocity_windows={"1h": "1h", "24h": "24h", "7d": "7d"},
    ratio_window="30d",
    smurf_threshold=10000.0
)

# Apply the transformation to the enriched transactions DataFrame
df_features = feature_pipeline.transform(trans_enriched_df)

# Display the schema and some data with the new features
print("Schema do DataFrame com Features:")
df_features.printSchema()

print("Primeiras 5 linhas do DataFrame com Features:")
df_features.show(5)

Schema do DataFrame com Features:
root
 |-- Timestamp: timestamp (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- From Account: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- To Account: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)
 |-- From Bank Name: string (nullable = true)
 |-- From Entity ID: string (nullable = true)
 |-- From Entity Name: string (nullable = true)
 |-- To Bank Name: string (nullable = true)
 |-- To Entity ID: string (nullable = true)
 |-- To Entity Name: string (nullable = true)
 |-- txn_count_1h_velocity: double (nullable = false)
 |-- amount_sum_1h_velocity: double (nullable = false)
 |-- amount_mean_1h_velocity: double (nullable = false)
 |-- amount_max_1h_velocity: d

In [ ]:
# df_features.write.parquet('/content/drive/MyDrive/TCC/data/features', mode='overwrite')
# print("df_features salvo em parquet.")

In [4]:
df_features = spark.read.parquet('/content/drive/MyDrive/TCC/data/features')

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

# EDA

### Análise da Variável Alvo: `Is Laundering`

In [ ]:
print("Distribuição da variável alvo 'Is Laundering':")
df_features.groupBy("Is Laundering").count().show()

Distribuição da variável alvo 'Is Laundering':
+-------------+--------+
|Is Laundering|   count|
+-------------+--------+
|            0|31863008|
|            1|   35230|
+-------------+--------+



### Análise da `Amount Received` por `Is Laundering`

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
print("Estatísticas de 'Amount Received' por 'Is Laundering':")
df_features.groupBy("Is Laundering").agg(
    F.mean("Amount Received").alias("Mean Amount Received"),
    F.stddev("Amount Received").alias("StdDev Amount Received"),
    F.min("Amount Received").alias("Min Amount Received"),
    F.max("Amount Received").alias("Max Amount Received")
).show()

Estatísticas de 'Amount Received' por 'Is Laundering':
+-------------+--------------------+----------------------+-------------------+-------------------+
|Is Laundering|Mean Amount Received|StdDev Amount Received|Min Amount Received|Max Amount Received|
+-------------+--------------------+----------------------+-------------------+-------------------+
|            0|   6379497.069583804|  2.5888672647249527E9|             1.0E-6|8.15860932172761E12|
|            1| 5.311674507372348E7|   4.988977908002805E9|             2.8E-5| 9.0627007807088E11|
+-------------+--------------------+----------------------+-------------------+-------------------+



### Análise das Features de Velocidade (Exemplo: `txn_count_1h_velocity`)

In [ ]:
print("Estatísticas de 'txn_count_1h_velocity' por 'Is Laundering':")
df_features.groupBy("Is Laundering").agg(
    F.mean("txn_count_1h_velocity").alias("Mean Txn Count 1h"),
    F.stddev("txn_count_1h_velocity").alias("StdDev Txn Count 1h"),
    F.max("txn_count_1h_velocity").alias("Max Txn Count 1h")
).show()

Estatísticas de 'txn_count_1h_velocity' por 'Is Laundering':
+-------------+------------------+-------------------+----------------+
|Is Laundering| Mean Txn Count 1h|StdDev Txn Count 1h|Max Txn Count 1h|
+-------------+------------------+-------------------+----------------+
|            0| 162.1870955497987|  662.7356778169013|         10909.0|
|            1|206.80706783990917|  741.6951238929706|         10193.0|
+-------------+------------------+-------------------+----------------+



### Análise das Features de Smurfing (Exemplo: `smurf_txn_count_24h_behavioral`)

In [ ]:
print("Estatísticas de 'smurf_txn_count_24h_behavioral' por 'Is Laundering':")
df_features.groupBy("Is Laundering").agg(
    F.mean("smurf_txn_count_24h_behavioral").alias("Mean Smurf Txn Count 24h"),
    F.stddev("smurf_txn_count_24h_behavioral").alias("StdDev Smurf Txn Count 24h"),
    F.max("smurf_txn_count_24h_behavioral").alias("Max Smurf Txn Count 24h")
).show()

Estatísticas de 'smurf_txn_count_24h_behavioral' por 'Is Laundering':
+-------------+------------------------+--------------------------+-----------------------+
|Is Laundering|Mean Smurf Txn Count 24h|StdDev Smurf Txn Count 24h|Max Smurf Txn Count 24h|
+-------------+------------------------+--------------------------+-----------------------+
|            0|       72.68752460533544|         296.4559441726601|                 2590.0|
|            1|       92.97715015611695|         330.7150875290676|                 2578.0|
+-------------+------------------------+--------------------------+-----------------------+



Total de registros: 31898238
Timestamp para o corte (80% dos dados): 2022-09-14 05:46:00
Registros no conjunto de treino (80%): 25519347
Registros no conjunto OOT (20%): 6378891

Intervalo de datas do conjunto de treino:
+-------------------+-------------------+
|      Min Timestamp|      Max Timestamp|
+-------------------+-------------------+
|2022-09-01 00:00:00|2022-09-14 05:46:00|
+-------------------+-------------------+


Intervalo de datas do conjunto OOT:
+-------------------+-------------------+
|      Min Timestamp|      Max Timestamp|
+-------------------+-------------------+
|2022-09-14 05:47:00|2022-09-28 15:58:00|
+-------------------+-------------------+



In [ ]:
# from pyspark.sql import SparkSession
# import os

# # 1. Definir a memória máxima para o driver (essencial no Colab)
# # Deixamos uma margem de segurança para o Sistema Operacional (~4-5GB)
# memory_limit = "46g"

# spark = SparkSession.builder \
#     .appName("Max_Performance_Spark") \
#     .config("spark.driver.memory", memory_limit) \
#     .config("spark.executor.memory", memory_limit) \
#     .config("spark.driver.maxResultSize", "10g") \
#     .config("spark.sql.shuffle.partitions", "200") \
#     .config("spark.memory.fraction", "0.8") \
#     .config("spark.memory.storageFraction", "0.3") \
#     .config("spark.ui.port", "4050") \
#     .getOrCreate()

# print(f"SparkSession inicializada com foco em High-RAM ({memory_limit}).")

In [ ]:
# # df_train.write.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_train', mode='overwrite')
# # print("df_train salvo em parquet.")
# df_oot.write.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_oot', mode='overwrite')
# print("df_oot salvo em parquet.")

df_oot salvo em parquet.


In [ ]:
# df_train = spark.read.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_train')
# df_oot = spark.read.parquet('/content/drive/MyDrive/TCC/data/treino_e_oot/df_oot')


Treinamento do modelo

In [6]:
from pyspark.sql import SparkSession

# Ajustado para os 167GB reais da sua instância
memory_limit = "140g"

spark = SparkSession.builder \
    .appName("TCC_AML_HighPerformance") \
    .config("spark.driver.memory", memory_limit) \
    .config("spark.executor.memory", memory_limit) \
    .config("spark.driver.maxResultSize", "30g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

print(f"Spark inicializado com {memory_limit} de RAM disponível.")

Spark inicializado com 140g de RAM disponível.


In [7]:
df_features = spark.read.parquet('/content/drive/MyDrive/TCC/data/features')

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StandardScaler, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, FloatType

# This section identifies features and builds the preprocessing pipeline.
# It needs to be executed before the main training loop.

target_col_name = 'Is Laundering'
initial_cols_to_remove = [
    target_col_name,
    'Timestamp',
    'From Bank', 'To Bank',
    'From Account', 'To Account',
    'From Entity ID', 'To Entity ID',
]

# Create a dummy DataFrame to infer schema for preprocessing pipeline construction
# This is done here to allow the pipeline to be defined outside the training loop,
# using the schema of df_features after removing known non-feature columns.

df_features_for_schema_inference = df_features.drop(*[col_name for col_name in initial_cols_to_remove if col_name in df_features.columns])

# Identify categorical and numerical columns using the schema
categorical_cols = [f.name for f in df_features_for_schema_inference.schema.fields if isinstance(f.dataType, StringType)]
numeric_cols = [f.name for f in df_features_for_schema_inference.schema.fields if isinstance(f.dataType, (IntegerType, LongType, DoubleType, FloatType))]

print("="*80)
print("IDENTIFICAÇÃO DE COLUNAS PARA PRÉ-PROCESSAMENTO")
print("="*80)

print(f"\n📊 Colunas categóricas ({len(categorical_cols)}):")
low_cardinality_cats = []
high_cardinality_cats = []

# For the purpose of pipeline construction, we'll assume a reasonable set of low cardinality categories.
# In a production setting, these would be explicitly defined or calculated dynamically on a sampled dataset.
# The current approach based on column names is a pragmatic choice given the available information.
for col_name in categorical_cols:
    if col_name in ['Receiving Currency', 'Payment Currency', 'Payment Format']:
        low_cardinality_cats.append(col_name)
    else:
        high_cardinality_cats.append(col_name)

for col_name in low_cardinality_cats:
    print(f"   - {col_name}")

print(f"\n📊 Categóricas de alta cardinalidade (>20) - Removidas do pré-processamento para este pipeline:")
for col_name in high_cardinality_cats:
    print(f"   - {col_name} → REMOVIDA")

# Adjust categorical_cols to only include low_cardinality_cats for the pipeline
categorical_cols_for_pipeline = low_cardinality_cats

print(f"\n📊 Colunas numéricas ({len(numeric_cols)}):")
for col_name in numeric_cols[:10]:
    print(f"   - {col_name}")
if len(numeric_cols) > 10:
    print(f"   ... e mais {len(numeric_cols) - 10} colunas")


print("\n" + "="*80)
print("CONSTRUINDO PIPELINE DE PRÉ-PROCESSAMENTO (PYSPARK ML)")
print("="*80)

# 1. Pipeline para features numéricas
# Imputação para colunas numéricas
numeric_imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=[f"{col}_imputed" for col in numeric_cols],
    strategy="median"
)

# Assemblar as features numéricas imputadas em um único vetor
numeric_assembler = VectorAssembler(
    inputCols=[f"{col}_imputed" for col in numeric_cols],
    outputCol="numeric_features",
    handleInvalid="keep"
)

# Escalonar as features numéricas
numeric_scaler = StandardScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric_features",
    withStd=True, withMean=True
)

# 2. Pipeline para features categóricas
categorical_stages = []
ohe_categorical_cols = []

for col_name in categorical_cols_for_pipeline:
    # StringIndexer para converter categorias string em índices numéricos
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_indexed",
        handleInvalid="keep" # Trata valores inválidos/nulos como uma nova categoria
    )
    categorical_stages.append(indexer)

    # OneHotEncoder para converter índices numéricos em vetores one-hot
    encoder = OneHotEncoder(
        inputCols=[f"{col_name}_indexed"],
        outputCols=[f"{col_name}_encoded"],
        dropLast=False # Mantém todas as categorias, similar a sparse_output=False do sklearn
    )
    categorical_stages.append(encoder)
    ohe_categorical_cols.append(f"{col_name}_encoded")

# 3. Final Vector Assembler para combinar todas as features
# Todas as features processadas (numéricas escalonadas + categóricas one-hot encoded)
final_feature_cols = ["scaled_numeric_features"] + ohe_categorical_cols
final_assembler = VectorAssembler(
    inputCols=final_feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

# Criar o pipeline completo do PySpark ML
preprocessor_spark = Pipeline(stages=[
    numeric_imputer,
    numeric_assembler,
    numeric_scaler
] + categorical_stages + [final_assembler])


print(f"\n📦 Transformadores:")
print(f"   - Numérico: Imputer (median) + VectorAssembler + StandardScaler")
print(f"   - Categórico: StringIndexer (handleInvalid='keep') + OneHotEncoder")
print(f"   - Final: VectorAssembler para combinar todas as features")

print(f"\n✓ Pipeline `preprocessor_spark` pronto para fit/transform!")


IDENTIFICAÇÃO DE COLUNAS PARA PRÉ-PROCESSAMENTO

📊 Colunas categóricas (7):
   - Receiving Currency
   - Payment Currency
   - Payment Format

📊 Categóricas de alta cardinalidade (>20) - Removidas do pré-processamento para este pipeline:
   - From Bank Name → REMOVIDA
   - From Entity Name → REMOVIDA
   - To Bank Name → REMOVIDA
   - To Entity Name → REMOVIDA

📊 Colunas numéricas (26):
   - Amount Received
   - Amount Paid
   - txn_count_1h_velocity
   - amount_sum_1h_velocity
   - amount_mean_1h_velocity
   - amount_max_1h_velocity
   - txn_count_24h_velocity
   - amount_sum_24h_velocity
   - amount_mean_24h_velocity
   - amount_max_24h_velocity
   ... e mais 16 colunas

CONSTRUINDO PIPELINE DE PRÉ-PROCESSAMENTO (PYSPARK ML)

📦 Transformadores:
   - Numérico: Imputer (median) + VectorAssembler + StandardScaler
   - Categórico: StringIndexer (handleInvalid='keep') + OneHotEncoder
   - Final: VectorAssembler para combinar todas as features

✓ Pipeline `preprocessor_spark` pronto para fi

In [10]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier

print("="*80)
print("CONFIGURANDO MODELOS PYSPARK ML")
print("="*80)

# Todos os classificadores suportam `weightCol` no PySpark 3.x para lidar com o desbalanceamento de classes.
spark_models = {
    'Logistic Regression': LogisticRegression(featuresCol='features', labelCol='label', weightCol='weight', elasticNetParam=0.0, regParam=0.0, maxIter=100),
    'Decision Tree': DecisionTreeClassifier(featuresCol='features', labelCol='label', weightCol='weight', maxDepth=10),
    # 'Random Forest': RandomForestClassifier(featuresCol='features', labelCol='label', weightCol='weight', numTrees=100, maxDepth=10),
    # 'GBTClassifier': GBTClassifier(featuresCol='features', labelCol='label', weightCol='weight', maxDepth=5, maxIter=100)
}

print(f"\n✅ {len(spark_models)} modelos configurados para PySpark ML.")
for model_name in spark_models.keys():
    print(f"   - {model_name} (com weightCol)")


CONFIGURANDO MODELOS PYSPARK ML

✅ 2 modelos configurados para PySpark ML.
   - Logistic Regression (com weightCol)
   - Decision Tree (com weightCol)


In [ ]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pandas as pd

print("="*80)
print("INICIANDO JANELA DESLIZANTE (EXPANDING WINDOW)")
print("="*80)

# 1. Obter min e max Timestamp do DataFrame completo
min_ts_df = df_features.agg(F.min('Timestamp')).collect()[0][0]
max_ts_df = df_features.agg(F.max('Timestamp')).collect()[0][0]

# Certificar que min_ts_df e max_ts_df são objetos datetime
if isinstance(min_ts_df, str): # Converter se for string
    min_ts_df = datetime.strptime(min_ts_df, '%Y-%m-%d %H:%M:%S')
if isinstance(max_ts_df, str): # Converter se for string
    max_ts_df = datetime.strptime(max_ts_df, '%Y-%m-%d %H:%M:%S')

total_duration = max_ts_df - min_ts_df
num_folds = 5 # Conforme requisito do usuário

# Dividir o período total em num_folds + 1 segmentos (um para cada 'mês' conceitual)
# Ex: 5 folds -> 6 segmentos. Fold k treina nos primeiros k+1 segmentos, testa no (k+2)-ésimo segmento.
period_duration = total_duration / (num_folds + 1)

print(f"Período total de dados: {min_ts_df} a {max_ts_df}")
print(f"Duração total: {total_duration}")
print(f"Duração de cada período (segmento): {period_duration}")

# Dicionário para armazenar as métricas de cada modelo por fold
metrics_results = {
    model_name: {'roc_auc': [], 'pr_auc': [], 'f1': [], 'precision': [], 'recall': [], 'accuracy': []}
    for model_name in spark_models.keys()
}

# Loop para a Janela Deslizante
for i in range(num_folds):
    print(f"\n========================================================================================")
    print(f"                                   Processando Fold {i+1}/{num_folds}                                  ")
    print(f"========================================================================================")

    # Define os limites de tempo para o treino e teste deste fold
    # Treino: [min_ts_df, min_ts_df + (i + 1) * period_duration)
    # Teste:  [min_ts_df + (i + 1) * period_duration, min_ts_df + (i + 2) * period_duration)

    train_end_ts_fold = min_ts_df + (i + 1) * period_duration
    test_start_ts_fold = train_end_ts_fold
    test_end_ts_fold = min_ts_df + (i + 2) * period_duration

    # Ajustar o final do último período de teste para incluir o max_ts_df
    if i == num_folds - 1:
        test_end_ts_fold = max_ts_df + timedelta(seconds=1) # Buffer para incluir o último timestamp

    print(f"Fold {i+1} - Período de Treino: {min_ts_df} a {train_end_ts_fold}")
    print(f"Fold {i+1} - Período de Teste:  {test_start_ts_fold} a {test_end_ts_fold}")

    # Filtrar os DataFrames de treino e teste
    df_train_fold_raw = df_features.filter(F.col('Timestamp') >= min_ts_df).filter(F.col('Timestamp') < train_end_ts_fold)
    df_test_fold_raw = df_features.filter(F.col('Timestamp') >= test_start_ts_fold).filter(F.col('Timestamp') < test_end_ts_fold)

    # Verificar se os DataFrames não estão vazios
    if df_train_fold_raw.count() == 0:
        print(f"WARNING: Dados de treino para o Fold {i+1} estão vazios. Pulando este fold.")
        continue
    if df_test_fold_raw.count() == 0:
        print(f"WARNING: Dados de teste para o Fold {i+1} estão vazios. Pulando este fold.")
        continue

    # SEPARAÇÃO DE FEATURES E TARGET para o fold atual
    target_col = 'Is Laundering'
    cols_to_remove = [
        target_col,
        'Timestamp',
        'From Bank', 'To Bank',
        'From Account', 'To Account',
        'From Entity ID', 'To Entity ID',
    ]
    cols_to_remove = [col for col in cols_to_remove if col in df_train_fold_raw.columns]

    # X_train_fold and X_test_fold are used for fitting the preprocessor (features only)
    X_train_fold_features_only = df_train_fold_raw.drop(*cols_to_remove)
    X_test_fold_features_only = df_test_fold_raw.drop(*cols_to_remove)

    print(f"\n📊 Fold {i+1} - Registros de Treino: {df_train_fold_raw.count()}, Registros de Teste: {df_test_fold_raw.count()}")

    # Fit preprocessor on X_train_fold_features_only only (evitando data leakage)
    print(f"\n⏳ Fold {i+1} - Aplicando fit do preprocessor_spark nos dados de treino (apenas features)...")
    model_spark_preprocessor_fold = preprocessor_spark.fit(X_train_fold_features_only)
    print(f"✓ Fold {i+1} - Preprocessor fit concluído!")

    print(f"\n⏳ Fold {i+1} - Transformando dados de treino e teste (mantendo a coluna target)...")
    # Apply transform to the raw dataframes (which still contain the target_col) and rename it to 'label'
    training_df_fold = model_spark_preprocessor_fold.transform(df_train_fold_raw).withColumnRenamed(target_col, "label")
    oot_df_fold = model_spark_preprocessor_fold.transform(df_test_fold_raw).withColumnRenamed(target_col, "label")
    print(f"✓ Fold {i+1} - Transformação concluída!")

    # Balanceamento de Classes (Via `weightCol` para efeito SMOTE-like)
    print(f"\n📊 Fold {i+1} - Calculando pesos para balanceamento de classes...")
    label_1_count_train_fold = training_df_fold.filter(F.col('label') == 1).count()
    label_0_count_train_fold = training_df_fold.filter(F.col('label') == 0).count()

    if label_1_count_train_fold == 0 or label_0_count_train_fold == 0:
        print(f"⚠️  Fold {i+1}: Uma das classes está vazia no conjunto de treino. Não é possível calcular pesos para balanceamento. Usando pesos iguais.")
        training_with_weights_fold_df = training_df_fold.withColumn("weight", F.lit(1.0))
        weight_for_0 = 1.0
        weight_for_1 = 1.0
    else:
        weight_for_1 = float(label_0_count_train_fold) / label_1_count_train_fold
        weight_for_0 = 1.0
        training_with_weights_fold_df = training_df_fold.withColumn(
            "weight",
            F.when(F.col("label") == 1, weight_for_1).otherwise(weight_for_0)
        )
    print(f"   Fold {i+1} - Pesos aplicados: label 0 = {weight_for_0:.2f}, label 1 = {weight_for_1:.2f}")

    # Treinar e Avaliar Modelos
    for model_name, classifier in spark_models.items():
        print(f"\n🔄 Fold {i+1} - Treinando e avaliando {model_name}...")

        # Treinar modelo usando os dados de treino com pesos
        model = classifier.fit(training_with_weights_fold_df)

        # Fazer previsões nos dados de teste do fold
        predictions = model.transform(oot_df_fold)

        # Avaliar ROC AUC e PR AUC
        evaluator_roc_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
        evaluator_pr_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")

        roc_auc = evaluator_roc_auc.evaluate(predictions)
        pr_auc = evaluator_pr_auc.evaluate(predictions)

        # Calcular outras métricas manualmente (Precision, Recall, F1, Accuracy)
        tp = predictions.filter((F.col("label") == 1) & (F.col("prediction") == 1)).count()
        fp = predictions.filter((F.col("label") == 0) & (F.col("prediction") == 1)).count()
        fn = predictions.filter((F.col("label") == 1) & (F.col("prediction") == 0)).count()
        tn = predictions.filter((F.col("label") == 0) & (F.col("prediction") == 0)).count()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0.0

        # Armazenar as métricas
        metrics_results[model_name]['roc_auc'].append(roc_auc)
        metrics_results[model_name]['pr_auc'].append(pr_auc)
        metrics_results[model_name]['f1'].append(f1)
        metrics_results[model_name]['precision'].append(precision)
        metrics_results[model_name]['recall'].append(recall)
        metrics_results[model_name]['accuracy'].append(accuracy)

        print(f"  ✅ Fold {i+1} - {model_name} - Resultados (Teste):")
        print(f"     ROC AUC: {roc_auc:.4f}")
        print(f"     PR AUC: {pr_auc:.4f}")
        print(f"     Precision: {precision:.4f}")
        print(f"     Recall (Sensitividade): {recall:.4f}")
        print(f"     F1-Score: {f1:.4f}")
        print(f"     Accuracy: {accuracy:.4f}")
        print(f"     Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    # Limpar referências e cache para otimização de memória
    # Despersiste os DataFrames intermediários para liberar memória no cluster Spark
    df_train_fold_raw.unpersist()
    df_test_fold_raw.unpersist()
    X_train_fold_features_only.unpersist() # Changed from X_train_fold
    # y_train_fold.unpersist() # No longer needed separately
    X_test_fold_features_only.unpersist() # Changed from X_test_fold
    # y_test_fold.unpersist() # No longer needed separately
    training_df_fold.unpersist()
    oot_df_fold.unpersist()
    training_with_weights_fold_df.unpersist()
    # X_train_transformed_fold.unpersist() # No longer explicitly created this way
    # X_test_transformed_fold.unpersist() # No longer explicitly created this way
    if 'predictions' in locals(): # Verifica se predictions foi criada antes de despersistir
        predictions.unpersist()

print("\n" + "="*80)
print("RESULTADOS FINAIS DA JANELA DESLIZANTE (EXPANDING WINDOW)")
print("="*80)

final_summary_data = []

for model_name, metrics in metrics_results.items():
    for metric_name, values in metrics.items():
        if values:
            mean_val = pd.Series(values).mean()
            std_val = pd.Series(values).std()
            final_summary_data.append([model_name, metric_name.replace('_', ' ').upper(), mean_val, std_val])
        else:
            final_summary_data.append([model_name, metric_name.replace('_', ' ').upper(), None, None])

final_summary_df = pd.DataFrame(final_summary_data, columns=['Model', 'Metric', 'Mean', 'Std Dev'])
print(final_summary_df.to_string())

# Encontrar o melhor modelo baseado na média do PR AUC
mean_pr_auc_scores = {model_name: pd.Series(metrics_results[model_name]['pr_auc']).mean()
                      for model_name in spark_models.keys() if metrics_results[model_name]['pr_auc']}

if mean_pr_auc_scores:
    best_model_overall = max(mean_pr_auc_scores, key=mean_pr_auc_scores.get)
    print(f"\n🏆 Melhor Modelo (Média PR AUC): {best_model_overall}")
    print(f"   Média PR AUC: {mean_pr_auc_scores[best_model_overall]:.4f}")
else:
    print("\n⚠️ Não foi possível determinar o melhor modelo pois não há métricas de PR AUC válidas.")

print("\n" + "="*80)
print("✅ TREINAMENTO E AVALIAÇÃO CONCLUÍDOS")
print("="*80)


INICIANDO JANELA DESLIZANTE (EXPANDING WINDOW)
Período total de dados: 2022-09-01 00:00:00 a 2022-09-28 15:58:00
Duração total: 27 days, 15:58:00
Duração de cada período (segmento): 4 days, 14:39:40

                                   Processando Fold 1/5                                  
Fold 1 - Período de Treino: 2022-09-01 00:00:00 a 2022-09-05 14:39:40
Fold 1 - Período de Teste:  2022-09-05 14:39:40 a 2022-09-10 05:19:20

📊 Fold 1 - Registros de Treino: 10350203, Registros de Teste: 9366181

⏳ Fold 1 - Aplicando fit do preprocessor_spark nos dados de treino (apenas features)...
✓ Fold 1 - Preprocessor fit concluído!

⏳ Fold 1 - Transformando dados de treino e teste (mantendo a coluna target)...
✓ Fold 1 - Transformação concluída!

📊 Fold 1 - Calculando pesos para balanceamento de classes...
   Fold 1 - Pesos aplicados: label 0 = 1.00, label 1 = 1613.44

🔄 Fold 1 - Treinando e avaliando Logistic Regression...
  ✅ Fold 1 - Logistic Regression - Resultados (Teste):
     ROC AUC: 0.91

#SHAP

In [ ]:
# from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
# from pyspark.sql.functions import col, lit, when, ceil, log
# from pyspark.sql.window import Window
# import pandas as pd
# from pyspark.sql.types import IntegerType, ArrayType, DoubleType # Added ArrayType, DoubleType

# print("\n" + "="*80)
# print("CÁLCULO DE KS E PSI")
# print("="*80)

# # --- 1. Obter o melhor modelo e gerar previsões para Treino e OOT ---
# # Reconstruir o dicionário de classificadores para obter o classificador do melhor modelo
# sPark_models_classifiers = {
#     'Logistic Regression': LogisticRegression(featuresCol='features', labelCol='label', elasticNetParam=0.0, regParam=0.0, maxIter=100),
#     'Decision Tree': DecisionTreeClassifier(featuresCol='features', labelCol='label', maxDepth=10)

# # Get the best model name from the previous results_pd (assuming it's sorted by PR AUC)
# # If results_pd is not available, you might need to re-run the training loop or manually set it.
# # For now, let's assume `best_model_name` is still in scope from previous execution.

# # If `results_pd` is not directly available or updated, let's find the best model again
# # This part assumes `results` dictionary from previous execution is available
# if 'results' in globals():
#     results_pd_temp = pd.DataFrame(results).T
#     results_pd_temp = results_pd_temp.sort_values('PR AUC', ascending=False)
#     best_model_name = results_pd_temp.index[0]
# else:
#     print("WARNING: `results` dictionary not found. Assuming 'Decision Tree' as best model for demonstration.")
#     best_model_name = 'Decision Tree' # Fallback if results are not in scope

# print(f"\n🏆 Melhor modelo (baseado em PR AUC): {best_model_name}")

# best_classifier = spark_models_classifiers[best_model_name]

# print("\n⏳ Re-treinando o melhor modelo para obter previsões completas...")
# best_model = best_classifier.fit(balanced_training_df)

# print("\n⏳ Gerando previsões para o conjunto de treino (referência PSI)...")
# training_predictions = best_model.transform(training_df).withColumn(
#     "probability_1", col("probability").cast(ArrayType(DoubleType()))[1] # Changed this line
# )

# print("\n⏳ Gerando previsões para o conjunto OOT...")
# oot_predictions = best_model.transform(oot_df).withColumn(
#     "probability_1", col("probability").cast(ArrayType(DoubleType()))[1] # Changed this line
# )
# print("✓ Previsões geradas com sucesso!")

# # --- 2. Cálculo do KS (Kolmogorov-Smirnov) para o conjunto OOT ---
# print("\n--- Calculando KS Statistic para OOT ---")

# # Criar uma coluna que rankeia as probabilidades para cálculo da CDF
# window_spec_prob = Window.orderBy(col("probability_1").asc())

# # Calcular CDF para a classe 0 e 1
# cum_dist = oot_predictions.withColumn(
#     "cum_pct_0",
#     F.sum(when(col("label") == 0, lit(1)).otherwise(lit(0))).over(window_spec_prob) / oot_predictions.filter(col("label") == 0).count()
# ).withColumn(
#     "cum_pct_1",
#     F.sum(when(col("label") == 1, lit(1)).otherwise(lit(0))).over(window_spec_prob) / oot_predictions.filter(col("label") == 1).count()
# )

# # Calcular a diferença absoluta e encontrar o máximo
# ks_statistic_row = cum_dist.select(F.max(F.abs(col("cum_pct_0") - col("cum_pct_1"))).alias("KS_Statistic")).collect()
# ks_statistic = ks_statistic_row[0]["KS_Statistic"]

# print(f"✅ KS Statistic (OOT): {ks_statistic:.4f}")

# # --- 3. Cálculo do PSI (Population Stability Index) para OOT vs Treino ---
# print("\n--- Calculando PSI (OOT vs Treino) ---")

# num_bins = 10 # Número de bins para o PSI

# # Calcular quantis para definir os limites dos bins no conjunto de treino
# quantiles = training_predictions.approxQuantile("probability_1", [float(i)/num_bins for i in range(num_bins + 1)], 0.01)

# def assign_bin(prob, quantiles_list):
#     # Atribuir o bin com base nos quantis
#     for i in range(len(quantiles_list) - 1):
#         if quantiles_list[i] <= prob <= quantiles_list[i+1]:
#             return i
#     return len(quantiles_list) - 2 # last bin for max value

# # Registrar a UDF
# assign_bin_udf = F.udf(lambda prob: assign_bin(prob, quantiles), IntegerType())

# # Atribuir bins às previsões de treino
# training_binned = training_predictions.withColumn("bin", assign_bin_udf(col("probability_1")))

# # Atribuir bins às previsões OOT
# oot_binned = oot_predictions.withColumn("bin", assign_bin_udf(col("probability_1")))

# # Contagem de cada bin para treino
# training_bin_counts = training_binned.groupBy("bin").count().withColumnRenamed("count", "train_count")

# # Contagem de cada bin para OOT
# oot_bin_counts = oot_binned.groupBy("bin").count().withColumnRenamed("count", "oot_count")

# # Juntar e calcular percentuais
# psi_df = training_bin_counts.join(oot_bin_counts, on="bin", how="full_outer") \
#     .fillna(0) \
#     .withColumn("train_pct", col("train_count") / training_predictions.count()) \
#     .withColumn("oot_pct", col("oot_count") / oot_predictions.count())

# # Calcular PSI para cada bin e somar
# psi_result_df = psi_df.withColumn("psi_component",
#     when( (col("train_pct") == 0) | (col("oot_pct") == 0), lit(0)) # Handle cases where a bin is empty in train or OOT
#     .otherwise((col("oot_pct") - col("train_pct")) * log(col("oot_pct") / col("train_pct")))
# )

# psi_statistic_row = psi_result_df.agg(F.sum("psi_component").alias("PSI_Statistic")).collect()
# psi_statistic = psi_statistic_row[0]["PSI_Statistic"]

# print(f"✅ PSI Statistic (OOT vs Treino): {psi_statistic:.4f}")

# print("\n" + "="*80)
# print("CÁLCULO DE KS E PSI CONCLUÍDO")
# print("="*80)


In [ ]:
# import shap
# import pandas as pd
# from sklearn.tree import DecisionTreeClassifier # Assuming Decision Tree was the best model
# import matplotlib.pyplot as plt

# print("\n" + "="*80)
# print("EXPLICABILIDADE DO MODELO COM SHAP")
# print("="*80)

# # --- 1. Extrair Nomes das Features ---
# # A metadata da coluna 'features' contém os nomes após o VectorAssembler
# feature_names = []
# feature_attrs = X_train_transformed_spark.schema["features"].metadata["ml_attr"]["attrs"]

# for attr_type in feature_attrs:
#     for feature_info in feature_attrs[attr_type]:
#         feature_names.append(feature_info['name'])

# print(f"Total de features para SHAP: {len(feature_names)}")
# print("Primeiras 10 features:\n", feature_names[:10])

# # --- 2. Preparar Dados para Scikit-learn (Amostragem para evitar OOM) ---
# print("\n⏳ Amostrando e convertendo dados para Pandas (para SHAP)...")

# # Amostrar o DataFrame Spark antes de converter para Pandas para evitar problemas de memória
# # Usaremos uma amostra maior para o treino do modelo SHAP, mas uma amostra menor para calcular os shap values
# # (a menos que o dataset seja pequeno o suficiente)

# sample_fraction_train = 0.1 # Ex: 10% do treino para treinar o modelo sklearn
# sample_fraction_oot_shap = 0.001 # Ex: 0.1% do OOT para calcular SHAP values

# # Converter apenas as colunas 'features' e 'label'
# training_df_pd_sample = balanced_training_df.select("features", "label").sample(False, sample_fraction_train, seed=42).toPandas()
# oot_df_pd_sample = oot_df.select("features", "label").sample(False, sample_fraction_oot_shap, seed=42).toPandas()

# X_train_pd = pd.DataFrame(training_df_pd_sample['features'].tolist(), columns=feature_names)
# y_train_pd = training_df_pd_sample['label']

# X_oot_pd_shap = pd.DataFrame(oot_df_pd_sample['features'].tolist(), columns=feature_names)
# y_oot_pd_shap = oot_df_pd_sample['label']

# print(f"Tamanho da amostra de treino para Sklearn: {len(X_train_pd)} linhas")
# print(f"Tamanho da amostra OOT para cálculo de SHAP: {len(X_oot_pd_shap)} linhas")

# # --- 3. Treinar um modelo Scikit-learn equivalente ---
# print("\n⏳ Treinando Decision Tree (sklearn) na amostra de treino...")
# sklearn_model = DecisionTreeClassifier(max_depth=10, random_state=42) # Usar mesmos parâmetros do melhor modelo PySpark
# sklearn_model.fit(X_train_pd, y_train_pd)
# print("✓ Modelo Scikit-learn treinado!")

# # --- 4. Calcular SHAP Values ---
# print("\n⏳ Calculando SHAP values (isso pode levar um tempo, dependendo do tamanho da amostra)...")
# explainer = shap.TreeExplainer(sklearn_model)
# shap_values = explainer.shap_values(X_oot_pd_shap)
# print("✓ SHAP values calculados!")

# # --- 5. Visualizar SHAP Results ---
# print("\n--- Visualizações SHAP ---")

# # Resumo global da importância das features
# print("Gerando Summary Plot...")
# shap.summary_plot(shap_values[1], X_oot_pd_shap, feature_names=feature_names, show=False)
# plt.title("SHAP Summary Plot (Classe Positiva)")
# plt.show()

# # Plot para uma única instância (opcional)
# # if len(X_oot_pd_shap) > 0:
# #     print("Gerando Force Plot para a primeira instância...")
# #     shap.initjs()
# #     display(shap.force_plot(explainer.expected_value[1], shap_values[1][0,:], X_oot_pd_shap.iloc[0,:], feature_names=feature_names))
# # else:
# #     print("Não há instâncias suficientes na amostra OOT para gerar Force Plot.")

# print("\n" + "="*80)
# print("EXPLICABILIDADE SHAP CONCLUÍDA")
# print("="*80)
